# 01 · ET raw → LSL alignment

Build the common ET-raw/LSL affine clock mapping and its QC tables. This is the shared time-axis prerequisite for all three publication exports.

In [1]:
import importlib
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

for PROJECT_ROOT in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (PROJECT_ROOT / 'helpers').is_dir():
        sys.path.insert(0, str(PROJECT_ROOT))
        break
else:
    raise FileNotFoundError('Could not locate the BIGPROJECT/helpers folder')

from helpers.project_config import load_project_config
from helpers.sync_quality import et_lsl_alignment as alignment
importlib.reload(alignment)

CONFIG = load_project_config(PROJECT_ROOT / 'ImBrAid' / 'config.json')
PROJECT_NAME = CONFIG.project
DATA_ROOT = CONFIG.path('paths', 'processing_data_root')
ALIGNMENT_CSV = CONFIG.path('files', 'et_lsl_alignment_csv')
WINDOWS_CSV = CONFIG.path('files', 'et_lsl_windows_csv')
ISSUES_CSV = CONFIG.path('files', 'et_lsl_issues_csv')
OVERWRITE = True
PARTICIPANTS = None
VISITS = None
LEGACY_MAPPING_CSV = None
WINDOW_S = 60.0
STEP_S = 30.0
LOCAL_RESIDUAL_SEARCH_RADIUS_S = 10.0
MIN_GOOD_WINDOWS = 3
REVIEW_DRIFT_PPM = 2000.0
MAX_DRIFT_PPM = 10000.0

In [2]:
jobs = alignment.discover_et_lsl_jobs(
    DATA_ROOT, project=PROJECT_NAME, participants=PARTICIPANTS, visits=VISITS,
)
display(jobs[['pid', 'visit', 'discovery_status', 'et_candidate_count', 'xdf_candidate_count']])

,pid,visit,discovery_status,et_candidate_count,xdf_candidate_count
0,A7G5SM,VX,ready,1,1
1,ABVLGQ,VX,ready,1,1
2,B9NYQ8,VX,ready,1,1
3,CRVWF9,VX,ready,1,1
4,CRYSEP,VX,ready,1,1
5,CU6DM4,VX,ready,1,1
6,DUJL2L,VX,ready,1,1
7,E4MKXT,VX,ready,1,1
8,EASFXM,VX,ready,1,1
9,FEDZAG,VX,ready,1,1


In [3]:
legacy = (
    pd.read_csv(LEGACY_MAPPING_CSV, dtype={'pid': str, 'visit': str})
    if LEGACY_MAPPING_CSV is not None and Path(LEGACY_MAPPING_CSV).is_file()
    else None
)
alignments, windows, issues = alignment.build_et_lsl_alignments(
    jobs, legacy_results=legacy, window_s=WINDOW_S, step_s=STEP_S,
    local_residual_search_radius_s=LOCAL_RESIDUAL_SEARCH_RADIUS_S,
    min_good_windows=MIN_GOOD_WINDOWS, review_drift_ppm=REVIEW_DRIFT_PPM,
    max_drift_ppm=MAX_DRIFT_PPM,
)
if OVERWRITE:
    alignment.save_et_lsl_alignment_results(
        alignments, windows, issues, alignment_csv=ALIGNMENT_CSV,
        windows_csv=WINDOWS_CSV, issues_csv=ISSUES_CSV,
    )
display(alignments[[
    'pid', 'visit', 'mapping_status', 'usable', 'quality_grade',
    'review_required', 'drift_ppm', 'n_good_windows',
    'fit_residual_p95_s', 'mapping_reason', 'warnings',
]])
display(issues)

Stream 1: Calculated effective sampling rate 41.0292 Hz is different from specified rate 50.0000 Hz.
Stream 11: Calculated effective sampling rate 41.0292 Hz is different from specified rate 50.0000 Hz.
Stream 7: Calculated effective sampling rate 41.0292 Hz is different from specified rate 50.0000 Hz.
Stream 2: Calculated effective sampling rate 0.1998 Hz is different from specified rate 10.0000 Hz.
Stream 9: Calculated effective sampling rate 41.0292 Hz is different from specified rate 50.0000 Hz.
Stream 8: Calculated effective sampling rate 41.0292 Hz is different from specified rate 50.0000 Hz.
Stream 3: Calculated effective sampling rate 0.1997 Hz is different from specified rate 10.0000 Hz.
Stream 8: Calculated effective sampling rate 0.0000 Hz is different from specified rate 250.0000 Hz.
Stream 9: Calculated effective sampling rate 0.1997 Hz is different from specified rate 10.0000 Hz.
Stream 3: Calculated effective sampling rate 0.1997 Hz is different from specified rate 10.00

Stream 3: Calculated effective sampling rate 0.1997 Hz is different from specified rate 10.0000 Hz.
Stream 1: Calculated effective sampling rate 43.5574 Hz is different from specified rate 50.0000 Hz.
Stream 7: Calculated effective sampling rate 0.1999 Hz is different from specified rate 10.0000 Hz.
Stream 6: Calculated effective sampling rate 43.5574 Hz is different from specified rate 50.0000 Hz.
Stream 11: Calculated effective sampling rate 43.5574 Hz is different from specified rate 50.0000 Hz.
Stream 8: Calculated effective sampling rate 43.5574 Hz is different from specified rate 50.0000 Hz.
Stream 9: Calculated effective sampling rate 43.5574 Hz is different from specified rate 50.0000 Hz.
Stream 10: Calculated effective sampling rate 0.1997 Hz is different from specified rate 10.0000 Hz.
Stream 8: Calculated effective sampling rate 0.1999 Hz is different from specified rate 10.0000 Hz.
Stream 8: Calculated effective sampling rate 0.2224 Hz is different from specified rate 10.00

,pid,visit,mapping_status,usable,quality_grade,review_required,drift_ppm,n_good_windows,fit_residual_p95_s,mapping_reason,warnings
0,A7G5SM,VX,content_affine,True,B,False,-7.209940,69.0,0.019176,local content windows,residual p95 4.046 s exceeds limit | possible ...
1,ABVLGQ,VX,content_affine,True,A,False,9.429200,89.0,0.001292,local content windows,
2,B9NYQ8,VX,content_affine,True,A,False,9.171459,89.0,0.001561,local content windows,
3,CRVWF9,VX,content_affine,True,A,False,8.997442,88.0,0.001241,local content windows,
4,CRYSEP,VX,content_affine,True,A,False,10.443689,89.0,0.000934,local content windows,
5,CU6DM4,VX,content_affine,True,A,False,9.799556,93.0,0.001256,local content windows,
6,DUJL2L,VX,content_affine,True,A,False,10.948969,91.0,0.001274,local content windows,
7,E4MKXT,VX,content_affine,True,A,False,7.508684,89.0,0.001194,local content windows,
8,EASFXM,VX,content_affine,True,A,False,8.914502,89.0,0.001421,local content windows,
9,FEDZAG,VX,content_affine,True,A,False,975.412432,88.0,0.108333,local content windows,


,project,pid,visit,severity,code,message
0,ImBrAid,A7G5SM,VX,warning,fit_qc,residual p95 4.046 s exceeds limit
1,ImBrAid,A7G5SM,VX,warning,fit_qc,possible clock jump 4.674 s
2,ImBrAid,A7G5SM,VX,warning,fit_qc,residual p95 3.962 s exceeds limit
3,ImBrAid,A7G5SM,VX,warning,fit_qc,possible clock jump 2.417 s
4,ImBrAid,FGJK2B,VX,warning,fit_qc,residual p95 4.727 s exceeds limit
5,ImBrAid,FGJK2B,VX,warning,fit_qc,possible clock jump 9.374 s
6,ImBrAid,FGJK2B,VX,warning,fit_qc,residual p95 8.245 s exceeds limit
7,ImBrAid,FGJK2B,VX,warning,fit_qc,possible clock jump 13.752 s
8,ImBrAid,G93LYM,VX,warning,fit_qc,residual p95 2.610 s exceeds limit
9,ImBrAid,G93LYM,VX,warning,fit_qc,possible clock jump 9.107 s
